# Skin Cancer Detection using Custom CNN

## Problem Statement
Melanoma is a dangerous form of skin cancer that accounts for a majority of skin cancer deaths. Early detection using automated deep learning algorithms significantly improves patient survival rates.

This notebook demonstrates an end-to-end Convolutional Neural Network (CNN) pipeline built using **TensorFlow/Keras** following the standard ISIC / HAM10000 skin cancer detection workflow:

1. **Data Loading & Preprocessing**
2. **Exploratory Visualizations**
3. **Baseline Custom CNN Model Construction**
4. **Overfitting Diagnosis & Evaluation**
5. **Data Augmentation & Regularization Strategy**
6. **Class Imbalance Analysis**
7. **Handling Class Imbalance via Augmentor**
8. **Final Model Training & Confusion Matrix Evaluation**

---

## Step 1: Importing Required Libraries

In [ ]:
import os
import pathlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix
import Augmentor

print("TensorFlow Version:", tf.__version__)
print("GPU Devices:", tf.config.list_physical_devices('GPU'))

## Step 2: Data Loading & Dataset Pipeline Setup

In [ ]:
# Smart Data Directory Detection (Kaggle vs Local Environment)
if os.path.exists('/kaggle/input'):
    kaggle_inputs = list(pathlib.Path('/kaggle/input').rglob('*'))
    # Find directory containing image class subfolders
    candidate_dirs = [p for p in pathlib.Path('/kaggle/input').rglob('*') if p.is_dir() and len(list(p.glob('*.jpg'))) > 0]
    data_dir = candidate_dirs[0].parent if candidate_dirs else pathlib.Path('/kaggle/input')
    print(f"Detected Kaggle environment data path: {data_dir}")
else:
    data_dir = pathlib.Path("./data/processed")
    print(f"Detected local environment data path: {data_dir.resolve()}")

batch_size = 32
img_height = 180
img_width = 180
epochs = 20

In [ ]:
# Create training and validation datasets (80/20 split)
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f"\nDetected {num_classes} classes: {class_names}")

## Step 3: Exploratory Data Visualization

In [ ]:
# Visualize 9 sample images from the training dataset
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Configure dataset buffering for maximum GPU performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

## Step 4: Baseline Custom CNN Model Architecture

In [ ]:
# Build Baseline Convolutional Neural Network
model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(img_height, img_width, 3)),
    
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

model.summary()

## Step 5: Model Training & Overfitting Diagnosis

In [ ]:
# Train Baseline CNN Model
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs
)

In [ ]:
# Plot Baseline Training & Validation Accuracy and Loss
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Baseline Model Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Baseline Model Loss')
plt.show()

## Step 6: Data Augmentation & Regularization Strategy

In [ ]:
# Define Data Augmentation & Dropout Pipeline to prevent overfitting
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.1)
])

model_augmented = models.Sequential([
    layers.Input(shape=(img_height, img_width, 3)),
    data_augmentation,
    layers.Rescaling(1./255),
    
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Dropout(0.3),
    
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation='softmax')
])

model_augmented.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

model_augmented.summary()

In [ ]:
# Train Model with Data Augmentation & Dropout
history_augmented = model_augmented.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs
)

In [ ]:
# Plot Accuracy & Loss for Augmented Model
acc = history_augmented.history['accuracy']
val_acc = history_augmented.history['val_accuracy']
loss = history_augmented.history['loss']
val_loss = history_augmented.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Train Acc (Augmented)')
plt.plot(epochs_range, val_acc, label='Val Acc (Augmented)')
plt.legend(loc='lower right')
plt.title('Augmented Model Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Train Loss (Augmented)')
plt.plot(epochs_range, val_loss, label='Val Loss (Augmented)')
plt.legend(loc='upper right')
plt.title('Augmented Model Loss')
plt.show()

## Step 7: Class Imbalance Analysis

In [ ]:
# Inspect class distribution counts
class_counts = {}
for cls in class_names:
    cls_path = data_dir / cls
    if cls_path.exists():
        count = len(list(cls_path.glob('*.jpg'))) + len(list(cls_path.glob('*.jpeg')))
        class_counts[cls] = count

df_counts = pd.DataFrame(list(class_counts.items()), columns=['Class', 'Count'])
print("Class Distribution:")
print(df_counts)

plt.figure(figsize=(10, 5))
sns.barplot(x='Class', y='Count', data=df_counts, palette='viridis')
plt.title('Class Distribution Across Skin Lesions')
plt.xticks(rotation=45)
plt.show()

## Step 8: Handling Class Imbalance using Augmentor

In [ ]:
# Rebalance minority classes to 500 images per class using Augmentor
for cls in class_names:
    cls_dir = str(data_dir / cls)
    if os.path.exists(cls_dir):
        p = Augmentor.Pipeline(cls_dir)
        p.rotate(probability=0.7, max_left_rotate=10, max_right_rotate=10)
        p.zoom(probability=0.5, min_factor=1.1, max_factor=1.5)
        p.flip_left_right(probability=0.5)
        p.sample(500)

## Step 9: Final Model Evaluation & Confusion Matrix

In [ ]:
# Compute confusion matrix and classification report
y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model_augmented.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Final Confusion Matrix')
plt.show()

print("Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))